# 🪐 AstroOS — Kundli LLM Fine-Tuning Notebook (Google Colab Free T4 GPU)

This notebook fine-tunes **Meta-Llama-3.1-8B-Instruct** or **Qwen-2.5-7B** on authentic Shastric horoscope readings generated from AstroOS.

### Requirements:
- In Google Colab menu: **Runtime** -> **Change runtime type** -> Select **T4 GPU** (Free tier).
- Your `kundli_llm_sft_dataset.jsonl` file generated by `scripts/generate_kundli_llm_dataset.py`.

In [ ]:
# Step 1: Install Unsloth and training dependencies (Ultra-fast QLoRA)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
# Step 2: Upload the training dataset (kundli_llm_sft_dataset.jsonl)
from google.colab import files
import os

if not os.path.exists("kundli_llm_sft_dataset.jsonl"):
    print("Upload your 'kundli_llm_sft_dataset.jsonl' file:")
    uploaded = files.upload()
else:
    print("kundli_llm_sft_dataset.jsonl already present!")

In [ ]:
# Step 3: Load base model in 4-bit with Unsloth
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Attach LoRA PEFT adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("Model and LoRA adapters configured successfully!")

In [ ]:
# Step 4: Load and tokenize dataset using Llama-3.1 chat template
from datasets import load_dataset

dataset = load_dataset("json", data_files="kundli_llm_sft_dataset.jsonl", split="train")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Loaded {len(dataset)} training examples!")

In [ ]:
# Step 5: Run SFT Training with QLoRA
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,  # Increase to 120 or 200 for deeper training
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

trainer_stats = trainer.train()
print("Training completed successfully!")

In [ ]:
# Step 6: Test the trained model in plain, grounded English
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are a warm, wise, and grounded Astrologer communicating in clear, natural English."},
    {"role": "user", "content": "Please give me a reading: My Ascendant is Aries, Moon is in Taurus, and Jupiter is my Atmakaraka. I am currently running Saturn-Mercury period. How does my career and present timing look?"}
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=500, use_cache=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


In [ ]:
# Step 7: Export model to GGUF (for Ollama / Cloud hosting)
model.save_pretrained("astroos_kundli_lora")
tokenizer.save_pretrained("astroos_kundli_lora")

# Export to 4-bit GGUF
model.save_pretrained_gguf("astroos_kundli_gguf", tokenizer, quantization_method="q4_k_m")
print("Model saved to 'astroos_kundli_gguf' ready for download!")